In [4]:
! pip install --upgrade -q pandas numpy scikit-learn imbalanced-learn optuna plotly nbformat

In [5]:
from collections import Counter
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier

import optuna

In [6]:
# --- Target Class Info ---

# Case classification codes
DISCARDED = "5"
DENGUE = "10"
DENGUE_ALARM = "11"
DENGUE_SEVERE = "12"
CHIKUNGUNYA = "13"

# Case outcome (evolution) codes
DEATH_DENGUE = "2"
DEATH_OTHER = "3"

# --- Scheme Columns Info ---

# Codes
DOES_NOT_APPLY = "6"  # Pregnancy
IGNORED = "9"   # General
YEAR_CODE = 4 # Type of age code indicating years

GEOGRAPHIC_COLUMNS  = {"sigla_uf_residencia"}
DEMOGRAPHIC_COLUMNS  = {"idade_paciente", "sexo_paciente", "gestante_paciente", "raca_cor_paciente"}
DISEASES_COLUMNS = {
    "possui_doenca_autoimune", "possui_diabetes", "possui_doencas_hematologicas",
    "possui_hepatopatias", "possui_doenca_renal", "possui_hipertensao",
    "possui_doenca_acido_peptica"
}
SYMPTOMS_COLUMNS  = {
    "apresenta_febre", "apresenta_cefaleia", "apresenta_exantema",
    "apresenta_dor_costas", "apresenta_mialgia", "apresenta_vomito", 
    "apresenta_conjutivite", "apresenta_dor_retroorbital", "apresenta_artralgia", 
    "apresenta_artrite", "apresenta_leucopenia", "apresenta_petequias"
}
OTHER_COLUMNS  = {"prova_laco", "dias_sintomas_notificacao"}

ALL_COLUMNS = {
    *GEOGRAPHIC_COLUMNS,
    *DEMOGRAPHIC_COLUMNS,
    *DISEASES_COLUMNS,
    *SYMPTOMS_COLUMNS,
    *OTHER_COLUMNS
}
ALL_COLUMNS_SORTED = sorted(list(ALL_COLUMNS))

BINARY_COLUMNS = {
    "prova_laco", "sexo_paciente", *DISEASES_COLUMNS, *SYMPTOMS_COLUMNS
}
NUMERIC_COLUMNS = {
    "idade_paciente", "dias_sintomas_notificacao"
}
CATEGORICAL_COLUMNS = {
    "gestante_paciente", "raca_cor_paciente", "sigla_uf_residencia"
}


# --- Training/Testing Constants ---

RANDOM_STATE = 42
TEST_RATIO = 0.15
N_FOLDS = 5
N_CLASSES = 3

TARGET_NAMES = ["low_risk", "alarm", "severe"]
TARGET_NAMES_COARSE = ["low_risk", "high_risk"]
TARGET_NAMES_FINE = ["alarm", "severe"]

TARGET_LABEL_MAP = {name: idx for idx, name in enumerate(TARGET_NAMES)}
LABEL_TARGET_MAP = {idx: name for idx, name in enumerate(TARGET_NAMES)}

COARSE_LABEL_MAP = {0: 0, 1: 1, 2: 1}
FINE_LABEL_MAP = {1: 0, 2: 1}
FINE_LABEL_MAP_REVERSE = {0: 1, 1: 2}

In [8]:
df = pd.read_csv("../../data/3_gold/dataset-processed-rf.csv")

X = df.drop("class", axis=1)
y = df["class"]
y = y.map(TARGET_LABEL_MAP)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_RATIO, random_state=RANDOM_STATE, stratify=y
)

In [9]:
def get_coarse_fine_data(X, y):
    X_coarse = X.copy()
    y_coarse = y.copy()
    y_coarse = y_coarse.map(COARSE_LABEL_MAP)

    high_risk_mask = y.isin([1, 2])
    X_fine = X[high_risk_mask].copy()
    y_fine = y[high_risk_mask].copy()
    y_fine = y_fine.map(FINE_LABEL_MAP)

    return X_coarse, y_coarse, X_fine, y_fine

In [10]:
def predict_soft_cascade(model_coarse, model_fine, X, y):
    # Get Probabilities from Coarse
    probs_coarse = model_coarse.predict_proba(X)
    p_high_risk = probs_coarse[:, 1]
    
    # Get Probabilities from Fine
    probs_fine = model_fine.predict_proba(X) 
    p_severe_given_high = probs_fine[:, 1] # P(Severe | High)
        
    # P(Severe) = P(High) * P(Severe | High)
    p_severe_global = p_high_risk * p_severe_given_high
    
    # P(Alarm) = P(High) * (1 - P(Severe | High))
    p_alarm_global = p_high_risk * (1 - p_severe_given_high)
    
    # P(Low) = 1 - P(High)
    p_low_global = 1.0 - p_high_risk
    
    final_probs = np.vstack([p_low_global, p_alarm_global, p_severe_global]).T
    final_preds = np.argmax(final_probs, axis=1)

    return f1_score(y, final_preds, average='macro'), final_preds


def predict_hard_cascade(
    model_coarse, model_fine, X, y, 
    threshold_coarse=0.5, threshold_fine=0.5
):
    # Predict Coarse (0 = Low, 1 = High)
    probs_coarse = model_coarse.predict_proba(X)[:, 1]
    
    preds_coarse = (probs_coarse >= threshold_coarse).astype(int)
    final_preds = preds_coarse.copy()
    
    high_risk_indices = np.where(preds_coarse == 1)[0]
    
    if len(high_risk_indices) > 0:
        X_high_risk = X.iloc[high_risk_indices]
        
        # Predict Fine (0 = Alarm, 1 = Severe)
        probs_fine_local = model_fine.predict_proba(X_high_risk)[:, 1]
        preds_fine_local = (probs_fine_local >= threshold_fine).astype(int)
        
        # Map Fine predictions back to Global labels
        preds_fine_global = np.array([FINE_LABEL_MAP_REVERSE[p] for p in preds_fine_local])
        final_preds[high_risk_indices] = preds_fine_global

    return f1_score(y, final_preds, average='macro'), final_preds

In [11]:
FIXED_PARAMS = {
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": -1
}

In [12]:
def train_on_folds(params_coarse, params_fine, use_soft_cascade, threshold_coarse=None, threshold_fine=None):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = []

    for train_idx, valid_idx in skf.split(X_train_full, y_train_full):
        X_train, y_train = X_train_full.iloc[train_idx], y_train_full.iloc[train_idx]
        X_valid, y_valid = X_train_full.iloc[valid_idx], y_train_full.iloc[valid_idx]

        clf_coarse = RandomForestClassifier(**params_coarse)
        clf_fine = RandomForestClassifier(**params_fine)

        X_coarse, y_coarse, X_fine, y_fine = get_coarse_fine_data(X_train, y_train)

        try:
            clf_coarse.fit(X_coarse, y_coarse)
            clf_fine.fit(X_fine, y_fine)

            if use_soft_cascade:
                f1, _ = predict_soft_cascade(clf_coarse, clf_fine, X_valid, y_valid)
            else:
                f1, _ = predict_hard_cascade(
                    clf_coarse, clf_fine, X_valid, y_valid,
                    threshold_coarse, threshold_fine
                )
            
            scores.append(f1)
        except ValueError:
            return 0.0, 1.0
        
    return np.mean(scores), np.std(scores)


def objective(trial: optuna.trial.Trial):
    params_coarse = {
        "n_estimators": trial.suggest_int("n_estimators_coarse", 100, 500, step=50),
        "max_depth": trial.suggest_int("max_depth_coarse", 5, 50),
        "min_samples_split": trial.suggest_int("min_samples_split_coarse", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf_coarse", 1, 10),
        "max_features": trial.suggest_categorical("max_features_coarse", ["sqrt", "log2"]),
        **FIXED_PARAMS
    }
    params_fine = {
        "n_estimators": trial.suggest_int("n_estimators_fine", 100, 500, step=50),
        "max_depth": trial.suggest_int("max_depth_fine", 5, 50),
        "min_samples_split": trial.suggest_int("min_samples_split_fine", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf_fine", 1, 10),
        "max_features": trial.suggest_categorical("max_features_fine", ["sqrt", "log2"]),
        **FIXED_PARAMS
    }

    mode = trial.suggest_categorical("mode", choices=["soft", "hard"])

    threshold_coarse = None
    threshold_fine = None
    use_soft_cascade = (mode == "soft")
    
    if not use_soft_cascade:
        threshold_coarse = trial.suggest_float("threshold_coarse", low=0.25, high=0.55)
        threshold_fine = trial.suggest_float("threshold_fine", low=0.25, high=0.55)

    avg, _ = train_on_folds(params_coarse, params_fine, use_soft_cascade, threshold_coarse, threshold_fine)
    return avg

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, timeout=14400, show_progress_bar=True, n_jobs=-1)

best_trial = study.best_trial
print('Best F1:', best_trial.value)
print('Best Params:', best_trial.params)

# Best F1: 0.5559225981160016
# Best Params: {'n_estimators_coarse': 100, 'max_depth_coarse': 48, 'min_samples_split_coarse': 13, 'min_samples_leaf_coarse': 6, 'max_features_coarse': 'sqrt', 'n_estimators_fine': 150, 'max_depth_fine': 24, 'min_samples_split_fine': 4, 'min_samples_leaf_fine': 1, 'max_features_fine': 'sqrt', 'mode': 'soft'}

In [ ]:
import pickle
from pathlib import Path

output_dir = Path('../results/optimization/random_forest')
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / 'optuna_study.pkl', 'wb') as fout:
    pickle.dump(study.sampler, fout)

In [ ]:
import optuna.visualization as vis

display(vis.plot_param_importances(study))
display(vis.plot_optimization_history(study))

In [16]:
params_coarse = {
    'n_estimators': 100, 'max_depth': 48, 'min_samples_split': 13, 'min_samples_leaf': 6, 'max_features': 'sqrt',
    **FIXED_PARAMS
}
params_fine = {
    'n_estimators': 150, 'max_depth': 24, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 'sqrt',
    **FIXED_PARAMS
}

avg_f1_final, std_f1_final = train_on_folds(params_coarse, params_fine, use_soft_cascade=True)

In [17]:
print(f"Avg F1: {avg_f1_final}")
print(f"Std F1: {std_f1_final}")

Avg F1: 0.5559225981160016
Std F1: 0.0030810511243525534
